# Fase 1 — Auditoría de calidad y perfilamiento del dataset**Proyecto:** Rentabilidad de modelos de alquiler en San Vicente del Raspeig (Alicante)## Objetivo de esta faseEvaluar la fiabilidad del dataset unificado antes de analizarlo: dimensiones, nulos, duplicados, tipos de datos y marcas de calidad heredadas de la limpieza. Después, perfilar qué hay en cada mercado.## Contexto del dato`dataset_unificado.csv` concatena cinco fuentes limpias en una sola tabla larga. Cada fila es **una oferta de una vivienda**. Lo que cambia entre filas es el mercado y la unidad del precio, por eso el precio va en tres columnas excluyentes (`precio_eur_mes`, `precio_eur_noche`, `precio_eur_venta`).---

## 1. Carga de datos

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snssns.set_theme(style="white", palette="bright")pd.set_option("display.max_columns", 30)pd.set_option("display.width", 200)df = pd.read_csv("../limpio/dataset_unificado.csv", sep=";")df.head()

## 2. Diagnóstico de salud de datos### 2.1 Dimensiones**Pregunta:** ¿cuántas ofertas y cuántas variables tenemos?

In [ ]:
# TODO: muestra el shape del dataframe y la lista de columnas# Pista: df.shape, df.columns.tolist()

### 2.2 Tipos de datos**Pregunta:** ¿están todas las columnas en el tipo que les corresponde? Fíjate especialmente en las de precio y en `fecha_captura`.

In [ ]:
# TODO: revisa los dtypes. ¿Hay alguna numérica leída como texto?# Pista: df.dtypes, df.info()

### 2.3 Valores nulos**Pregunta:** ¿qué porcentaje de nulos hay por columna?Atención: aquí muchos nulos son **esperados y correctos**, no un problema. Una fila de venta tiene `precio_eur_mes` nulo porque no le corresponde. Distingue los nulos estructurales de los que sí son ausencia real de dato (por ejemplo, `m2` o `zona` que no se pudo capturar).

In [ ]:
# TODO: calcula el % de nulos por columna# Pista: (df.isna().sum() / len(df) * 100).round(1).sort_values(ascending=False)

In [ ]:
# TODO: ahora calcula los nulos POR MERCADO, para separar los estructurales de los reales# Pista: df.groupby("mercado").apply(lambda g: g.isna().mean() * 100).round(1)

**Escribe aquí tu conclusión:** ¿qué nulos son estructurales y cuáles indican una carencia real de la extracción?_(tu respuesta)_

### 2.4 Duplicados**Pregunta:** ¿quedan duplicados tras la limpieza?Recuerda que la deduplicación cruzada Idealista↔Fotocasa ya se hizo en `limpieza_alquiler_residencial.py` con una heurística (precio + habitaciones + m² ±2), porque los portales no publican la calle. Aquí comprobamos si quedó algo.

In [ ]:
# TODO: busca duplicados exactos y duplicados por combinación de campos clave# Pista: df.duplicated().sum() y df.duplicated(subset=["mercado","precio_eur_mes","habitaciones","m2"]).sum()

### 2.5 Marcas de calidadEn la limpieza se marcaron (no se borraron) las filas problemáticas, para poder decidir después si excluirlas.**Pregunta:** ¿cuántas hay de cada tipo y qué implican?

In [ ]:
# TODO: cuenta los valores de marca_calidad y mira un ejemplo de cada tipo# Pista: df["marca_calidad"].value_counts(dropna=False)

**Escribe aquí tu decisión:** ¿cuáles excluyes del análisis y por qué? Justifícalo para cada marca:| Marca | ¿Excluir? | Motivo ||---|---|---|| `m2_sospechoso` | | || `outlier_lujo` | | || `villa_no_comparable` | | || `fuera_del_municipio` | | || `clasificacion_heuristica` | | |

---## 3. Perfilamiento por mercado### 3.1 Volumen de oferta**Pregunta:** ¿cuántas ofertas hay de cada mercado y de qué fuentes vienen?

In [ ]:
# TODO: cuenta filas por mercado, y cruza mercado x fuente# Pista: df["mercado"].value_counts() y pd.crosstab(df["mercado"], df["fuente"])

### 3.2 Características de la vivienda**Pregunta:** ¿cómo se distribuyen habitaciones y m² en cada mercado? ¿Son parques de vivienda comparables entre sí?Esta pregunta importa más de lo que parece: si el parque turístico son villas de 6 dormitorios y el de alquiler son pisos de 3, comparar sus precios sin más sería un error.

In [ ]:
# TODO: describe habitaciones y m2 por mercado, y haz un boxplot# Pista: df.groupby("mercado")[["habitaciones","m2"]].describe()# Para el gráfico: sns.boxplot(data=df, x="mercado", y="m2")

### 3.3 Precios por mercado**Pregunta:** ¿cuál es la distribución de precios en cada mercado?Cuidado con las unidades: no mezcles €/mes con €/noche ni con € de compra. Haz un gráfico por unidad.

In [ ]:
# TODO: describe cada columna de precio por separado y haz un histograma de cada una

### 3.4 Precio por m²**Pregunta:** ¿cómo varía el precio/m² según el número de habitaciones en alquiler? ¿Y en venta?Añade percentiles (P25, P50, P75), no solo la media.

In [ ]:
# TODO: analiza precio_por_m2 por mercado y por habitaciones# Pista: df.groupby(["mercado","habitaciones"])["precio_por_m2"].agg(["count","median",#        lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])

### 3.5 Comparación entre portales**Pregunta:** ¿hay sesgo de precio entre Idealista y Fotocasa? Si una fuente publica sistemáticamente más caro, hay que saberlo antes de mezclarlas.

In [ ]:
# TODO: compara la mediana de precio_eur_mes entre idealista y fotocasa# ¿La diferencia se explica por el tamaño de las viviendas de cada muestra?

---## 4. El arquetipoTodo el análisis financiero se hace sobre un mismo arquetipo de vivienda para que la comparación sea justa: **piso de 3 habitaciones, 70-130 m²**. La columna `es_comparable_arquetipo` ya lo marca.**Pregunta:** ¿cuántas ofertas comparables hay en cada mercado y cuál es su precio mediano?

In [ ]:
# TODO: filtra por es_comparable_arquetipo y calcula n y mediana por mercado# Comprobación: deberías obtener 990 EUR/mes (alquiler), 292 EUR/mes (habitación)# y 237.950 EUR (compra). Si no te cuadra, revisa el filtro.

**Escribe aquí tu conclusión de la fase 1:** estado del dato, qué se excluye y qué muestra queda para analizar._(tu respuesta)_